# Practice: Make a Network Claim You Can Defend

**Computational Text Analysis Pack · Module 19 — Networks from Text**

*Technique:* Building a network, testing its stability, and stating what it shows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/08-text-reuse-and-networks/19c_practice_network_argument.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **Which words sit at the centre of health discourse in this corpus — and is
> "centre" a property of the discourse or of my parameter choices?**

Module 19 warned that a collocation network is a picture of your thresholds.
This notebook takes that seriously by testing it.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import networkx as nx, math, numpy as np
docs = [str(t) for t in df[(df["category"] == "health") & (df["language"] == "English")]["text"]]
print(f"{len(docs)} English health documents")

STOP = {"the","a","an","and","of","to","in","on","for","with","is","was","are",
        "were","be","been","it","its","this","that","as","at","by","from","has",
        "have","had","will","said","he","she","they","we","not","but","or"}

def build(window=4, min_joint=12, min_freq=40, top_edges=200):
    uni, pair = Counter(), Counter()
    for d in docs:
        t = [w for w in tokenise(d) if w not in STOP]
        uni.update(t)
        for i, w in enumerate(t):
            for j in range(i+1, min(i+window+1, len(t))):
                if w != t[j]: pair[tuple(sorted((w, t[j])))] += 1
    edges = []
    for (a, b), n in pair.items():
        if n < min_joint or uni[a] < min_freq or uni[b] < min_freq: continue
        edges.append((a, b, 14 + math.log2(2*n/(uni[a]+uni[b]))))
    edges.sort(key=lambda e: -e[2])
    G = nx.Graph()
    for a, b, w in edges[:top_edges]: G.add_edge(a, b, weight=w)
    return G

G = build()
print(f"network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
btw = nx.betweenness_centrality(G)
deg = nx.degree_centrality(G)
top = sorted(btw, key=btw.get, reverse=True)[:12]
print(f"{'word':<16}{'betweenness':>13}{'degree':>10}")
print("-" * 40)
for w in top:
    print(f"{w:<16}{btw[w]:>13.4f}{deg[w]:>10.3f}")

In [ ]:
# The stability test. Vary every parameter and see whether the 'centre' holds.
print(f"{'window':>7}{'min_joint':>11}{'edges':>7}{'nodes':>7}   top 3 by betweenness")
print("-" * 78)
rankings = []
for window in [3, 4, 6]:
    for mj in [8, 12, 20]:
        g = build(window=window, min_joint=mj)
        if g.number_of_nodes() < 5: continue
        b = nx.betweenness_centrality(g)
        t3 = sorted(b, key=b.get, reverse=True)[:3]
        rankings.append(set(t3))
        print(f"{window:>7}{mj:>11}{g.number_of_edges():>7}{g.number_of_nodes():>7}   {', '.join(t3)}")
if rankings:
    stable = set.intersection(*rankings)
    print(f"\nwords in the top 3 under EVERY setting: {sorted(stable) or '(none)'}")
    print("Only those are safe to describe as central. The rest are parameters.")

In [ ]:
plt.figure(figsize=(11, 8))
pos = nx.spring_layout(G, k=0.5, seed=5, weight="weight")
d = dict(G.degree())
nx.draw_networkx_edges(G, pos, alpha=.2, width=.7)
nx.draw_networkx_nodes(G, pos, node_size=[30+50*d[n] for n in G],
                       node_color=[btw[n] for n in G], cmap="viridis", alpha=.85)
nx.draw_networkx_labels(G, pos, font_size=7)
plt.title("Health collocation network (log-Dice edges)")
plt.axis("off"); plt.tight_layout()

## What this licenses

**Defensible:** "In a collocation network of English health articles (window ±4,
log-Dice edges, minimum joint frequency 12, top 200 edges), the words [stable
set] have the highest betweenness under every parameter setting tested."

**Not defensible:** "X is central to health discourse." Betweenness is a property
of a graph you constructed, and module 19a showed the graph changing shape with
the stop list alone.

**Also worth stating:** the network is undirected and unweighted by direction, so
it says nothing about which word leads to which. Module 5d's Delta P is the
measure that carries direction, and it does not appear in a graph like this.

### 🔧 Your turn 1

Rebuild the network for a different category and compare the stable central words. Do the two discourses share a centre?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Often they share the generic institutional vocabulary — government, people,
year — and differ only at the periphery, which is a useful negative result: the
centre of a collocation network built this way tends to be a property of the
genre rather than of the topic. Getting past that usually means comparing
against a reference corpus first, as module 4 does for keywords, and building
the network from the words that survived that comparison.

</details>